# InverNet Grid Regeneration
Runs InverNet (50 epochs) for 4 paper-figure methods × 2 seeds and saves `recon_grid_*.png`.

**Row order:** A_plain_vfl → S_sign_quant (128b) → H_vq_M16 (128b) → H_vq_K64 (48b)

**Attach datasets:**
- `results_final` dataset (your checkpoints)
- `isic-2019-256-cache` (numpy image cache + fold indices)
- `isic-2019` (GT and metadata CSVs)

**Download after run:** `/kaggle/working/figs/recon_grid_*.png`


In [ ]:
import subprocess, sys
for pkg in ['lpips', 'scikit-image', 'timm']:
    mod = pkg.replace('-', '_').replace('scikit_image', 'skimage')
    try:
        __import__(mod)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)
print('Environment ready.')

In [ ]:
import os, json, shutil, traceback
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import Dict, List, Any, Optional, Tuple
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision import transforms
from sklearn.cluster import KMeans
import timm

print(f'torch {torch.__version__}  cuda={torch.cuda.is_available()}')

In [ ]:
# ── FILL IN the dataset path below (check your Kaggle Data sidebar) ──
RESULTS_DATASET = "/kaggle/input/YOUR-RESULTS-DATASET-SLUG/results_final"

WORKING      = "/kaggle/working"
CKPT_DIR     = f"{WORKING}/checkpoints"
FIG_DIR      = f"{WORKING}/figs"
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(FIG_DIR,  exist_ok=True)

if not os.path.exists(RESULTS_DATASET):
    raise FileNotFoundError(f"Dataset not found: {RESULTS_DATASET}. Check the Data sidebar.")

ckpt_src = os.path.join(RESULTS_DATASET, 'checkpoints')
copied = 0
for fn in os.listdir(ckpt_src):
    if fn.endswith('.pt'):
        shutil.copy(os.path.join(ckpt_src, fn), CKPT_DIR)
        copied += 1
print(f'Copied {copied} checkpoints from dataset.')
print('Checkpoints:', sorted(os.listdir(CKPT_DIR)))

In [ ]:
CACHE_PATH = "/kaggle/input/datasets/taimurjahanzeb/isic-2019-256-cache/images_256.npy"
FOLD_PATH  = "/kaggle/input/datasets/taimurjahanzeb/isic-2019-256-cache/fold_indices.npz"
GT_CSV     = "/kaggle/input/datasets/andrewmvd/isic-2019/ISIC_2019_Training_GroundTruth.csv"
META_CSV   = "/kaggle/input/datasets/andrewmvd/isic-2019/ISIC_2019_Training_Metadata.csv"

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

@dataclass
class V9Config:
    input_size: int             = 224
    store_size: int             = 256
    batch_size: int             = 32
    fold: int                   = 0
    vq_proj_dim: int            = 128
    vq_num_subspaces: int       = 8
    vq_codebook_size: int       = 256
    vq_commitment_weight: float = 0.25
    vq_codebook_weight: float   = 1.0
    vq_warmup_epochs: int       = 3
    vq_use_kmeans_init: bool    = True
    kmeans_samples: int         = 4096
    rm_sigma_fwd: float         = 0.01
    rm_sigma_bwd: float         = 0.01
    usage_entropy_weight: float = 0.01
    seeds: Tuple[int, ...]      = (42, 43)
    num_workers: int            = 2

CFG = V9Config()
assert CFG.vq_proj_dim % CFG.vq_num_subspaces == 0
print('Config ready.')

In [ ]:
gt = pd.read_csv(GT_CSV)
gt = gt.drop(columns=['UNK'], errors='ignore')
CLASS_NAMES = gt.columns[1:].tolist()
NUM_CLASSES = len(CLASS_NAMES)

meta_df = pd.read_csv(META_CSV)
merged  = gt.merge(meta_df, on='image', how='inner')

bins       = list(range(0, 90, 5))
age_labels = [f'age_{b}_{b+4}' for b in bins[:-1]] + ['age_85plus']
merged['age_bin'] = pd.cut(
    merged['age_approx'], bins=bins + [np.inf], labels=age_labels, right=False
).astype(str)
merged.loc[merged['age_approx'].isna(), 'age_bin'] = 'age_unknown'
age_oh = pd.get_dummies(merged['age_bin']).astype(np.float32).values

site_col = ('anatom_site_general_challenge'
            if 'anatom_site_general_challenge' in merged.columns
            else 'anatom_site_general')
merged[site_col] = merged[site_col].fillna('unknown')
loc_oh = pd.get_dummies(merged[site_col]).astype(np.float32).values
merged['sex'] = merged['sex'].fillna('unknown')
sex_oh = pd.get_dummies(merged['sex']).astype(np.float32).values

meta_array   = np.concatenate([age_oh, loc_oh, sex_oh], axis=1).astype(np.float32)
META_DIM     = meta_array.shape[1]
labels_array = merged[CLASS_NAMES].values.astype(np.float32)
label_ints   = np.argmax(labels_array, axis=1)

class_counts = np.bincount(label_ints, minlength=NUM_CLASSES).astype(np.float32)
tail_classes = sorted(np.argsort(class_counts)[:4].tolist())

_fd       = np.load(FOLD_PATH)
train_ind = _fd[f'train_{CFG.fold}']
val_ind   = _fd[f'val_{CFG.fold}']
all_images = np.load(CACHE_PATH)

print(f'Classes: {CLASS_NAMES}')
print(f'Train: {len(train_ind)}  Val: {len(val_ind)}')
print(f'Tail classes: {[CLASS_NAMES[c] for c in tail_classes]}')

In [ ]:
SET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
SET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

val_transforms = transforms.Compose([
    transforms.CenterCrop(CFG.input_size),
    transforms.ToTensor(),
    transforms.Normalize(SET_MEAN, SET_STD),
])
print('Transforms defined.')

In [ ]:
class V8Backbone(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        self.net = timm.create_model('efficientnet_b0', pretrained=pretrained, num_classes=0)
        self.out_dim = 1280
    def forward(self, x):
        return self.net(x)


class V8Projection(nn.Module):
    def __init__(self, in_dim=1280, out_dim=128):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)
        self.bn     = nn.BatchNorm1d(out_dim)
    def forward(self, h):
        return self.bn(self.linear(h))


class V8ProductQuantizer(nn.Module):
    def __init__(self, proj_dim, num_subspaces, codebook_size, commitment_weight=0.25):
        super().__init__()
        assert proj_dim % num_subspaces == 0
        self.proj_dim = proj_dim
        self.M = num_subspaces
        self.K = codebook_size
        self.subdim = proj_dim // num_subspaces
        self.commitment_weight = commitment_weight
        self.codebooks = nn.Parameter(
            torch.randn(num_subspaces, codebook_size, self.subdim) * 0.02
        )
        self.register_buffer('usage_counts',
            torch.zeros(num_subspaces, codebook_size, dtype=torch.long))

    def forward(self, z):
        B, D = z.shape
        z_sub = z.view(B, self.M, self.subdim)
        distances = (
            z_sub.unsqueeze(2).pow(2).sum(-1)
            + self.codebooks.pow(2).sum(-1).unsqueeze(0)
            - 2.0 * torch.einsum('bmd,mkd->bmk', z_sub, self.codebooks)
        )
        indices = distances.argmin(dim=-1)
        q_sub = torch.gather(
            self.codebooks.unsqueeze(0).expand(B, -1, -1, -1),
            2,
            indices.unsqueeze(-1).unsqueeze(-1).expand(-1, -1, 1, self.subdim),
        ).squeeze(2)
        codebook_loss   = F.mse_loss(q_sub, z_sub.detach())
        commitment_loss = F.mse_loss(z_sub, q_sub.detach())
        vq_loss = codebook_loss + self.commitment_weight * commitment_loss
        q_sub_st = z_sub + (q_sub - z_sub).detach()
        q = q_sub_st.reshape(B, D)
        with torch.no_grad():
            for m in range(self.M):
                counts = torch.bincount(indices[:, m], minlength=self.K)
                self.usage_counts[m] += counts
        return q, indices, vq_loss

    @torch.no_grad()
    def kmeans_init_from_batch(self, z_samples, seed=0):
        N = z_samples.size(0)
        z_np_sub = z_samples.detach().cpu().numpy().reshape(N, self.M, self.subdim)
        new_cb = np.zeros((self.M, self.K, self.subdim), dtype=np.float32)
        for m in range(self.M):
            km = KMeans(n_clusters=self.K, init='k-means++',
                        n_init=3, max_iter=50, random_state=seed + m)
            km.fit(z_np_sub[:, m, :])
            new_cb[m] = km.cluster_centers_.astype(np.float32)
        self.codebooks.data.copy_(torch.from_numpy(new_cb).to(self.codebooks.device))


class PlainPassiveClient(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.backbone = V8Backbone(pretrained=True)
        self.passive_emb_dim = self.backbone.out_dim
    def forward(self, x, use_quantizer=True):
        h = self.backbone(x)
        return h, None, torch.zeros((), device=h.device), h


class ProjPassiveClient(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.backbone   = V8Backbone(pretrained=True)
        self.projection = V8Projection(self.backbone.out_dim, cfg.vq_proj_dim)
        self.passive_emb_dim = cfg.vq_proj_dim
    def forward(self, x, use_quantizer=True):
        h = self.backbone(x)
        z = self.projection(h)
        return z, None, torch.zeros((), device=z.device), z


class SignPassiveClient(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.backbone   = V8Backbone(pretrained=True)
        self.projection = V8Projection(self.backbone.out_dim, cfg.vq_proj_dim)
        self.passive_emb_dim = cfg.vq_proj_dim
    def forward(self, x, use_quantizer=True):
        h = self.backbone(x)
        z = self.projection(h)
        s = z.sign()
        s_ste = z + (s - z).detach()
        return s_ste, None, torch.zeros((), device=z.device), z


class RandSignPassiveClient(nn.Module):
    def __init__(self, cfg, init_seed=0):
        super().__init__()
        self.backbone = V8Backbone(pretrained=True)
        in_dim  = self.backbone.out_dim
        out_dim = cfg.vq_proj_dim
        gen = torch.Generator().manual_seed(int(init_seed))
        W = torch.randn(in_dim, out_dim, generator=gen) / (in_dim ** 0.5)
        self.register_buffer('proj_W', W)
        self.passive_emb_dim = out_dim
    def forward(self, x, use_quantizer=True):
        h = self.backbone(x)
        z = h @ self.proj_W
        s = z.sign()
        s_ste = z + (s - z).detach()
        return s_ste, None, torch.zeros((), device=z.device), z


class VQPassiveClient(nn.Module):
    def __init__(self, cfg, codebook_size=None, num_subspaces=None, commitment_weight=None):
        super().__init__()
        K    = codebook_size    if codebook_size    is not None else cfg.vq_codebook_size
        M    = num_subspaces    if num_subspaces    is not None else cfg.vq_num_subspaces
        beta = commitment_weight if commitment_weight is not None else cfg.vq_commitment_weight
        self.backbone   = V8Backbone(pretrained=True)
        self.projection = V8Projection(self.backbone.out_dim, cfg.vq_proj_dim)
        self.quantizer  = V8ProductQuantizer(
            proj_dim=cfg.vq_proj_dim, num_subspaces=M,
            codebook_size=K, commitment_weight=beta)
        self.passive_emb_dim = cfg.vq_proj_dim
    def forward(self, x, use_quantizer=True):
        h = self.backbone(x)
        z = self.projection(h)
        if not use_quantizer:
            return z, None, torch.zeros((), device=z.device), z
        q, idx, vq_loss = self.quantizer(z)
        return q, idx, vq_loss, z


def build_passive(spec, seed):
    cfg = CFG
    if spec.passive_type == 'plain':     return PlainPassiveClient(cfg)
    if spec.passive_type == 'proj':      return ProjPassiveClient(cfg)
    if spec.passive_type == 'sign':      return SignPassiveClient(cfg)
    if spec.passive_type == 'rand_sign': return RandSignPassiveClient(cfg, init_seed=seed)
    if spec.passive_type == 'vq':
        return VQPassiveClient(cfg,
            codebook_size=spec.vq_codebook_size,
            num_subspaces=spec.vq_num_subspaces,
            commitment_weight=spec.vq_commitment_weight)
    raise ValueError(f'unknown passive_type: {spec.passive_type}')

print('Model classes defined.')

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.c1 = nn.Conv2d(ch, ch, 3, 1, 1)
        self.b1 = nn.BatchNorm2d(ch)
        self.c2 = nn.Conv2d(ch, ch, 3, 1, 1)
        self.b2 = nn.BatchNorm2d(ch)
    def forward(self, x):
        return F.relu(x + self.b2(self.c2(F.relu(self.b1(self.c1(x))))))


class InverNetV9(nn.Module):
    def __init__(self, in_dim, out_resolution=64):
        super().__init__()
        self.out_resolution = out_resolution
        self.fc = nn.Linear(in_dim, 256 * 4 * 4)
        self.res_blocks = nn.Sequential(ResBlock(256), ResBlock(256))
        self.upsampler = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(128,  64, 4, 2, 1), nn.BatchNorm2d(64),  nn.ReLU(inplace=True),
            nn.ConvTranspose2d( 64,  32, 4, 2, 1), nn.BatchNorm2d(32),  nn.ReLU(inplace=True),
            nn.ConvTranspose2d( 32,   3, 4, 2, 1),
        )
        self.refine = nn.Sequential(
            nn.Conv2d(3, 16, 3, 1, 1), nn.ReLU(inplace=True),
            nn.Conv2d(16, 3, 3, 1, 1), nn.Tanh(),
        )
    def forward(self, x):
        h = self.fc(x).view(-1, 256, 4, 4)
        h = self.res_blocks(h)
        h = self.upsampler(h)
        return self.refine(h)

print('InverNetV9 defined.')

In [ ]:
def _torch_load(path):
    try:
        return torch.load(path, map_location=DEVICE, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=DEVICE)


@dataclass
class MethodSpec:
    name: str
    passive_type: str
    comm_bits: int
    vq_codebook_size: int       = 256
    vq_num_subspaces: int       = 8
    vq_commitment_weight: float = 0.25
    vq_use_kmeans_init: bool    = True
    invernet_epochs: int        = 50


def _build_passive_for_load(spec_dict, seed):
    s = MethodSpec(
        name=spec_dict['name'],
        passive_type=spec_dict['passive_type'],
        comm_bits=spec_dict['comm_bits'],
        vq_codebook_size=spec_dict.get('vq_codebook_size', 256),
        vq_num_subspaces=spec_dict.get('vq_num_subspaces', 8),
        vq_commitment_weight=spec_dict.get('vq_commitment_weight', 0.25),
        vq_use_kmeans_init=spec_dict.get('vq_use_kmeans_init', True),
        invernet_epochs=spec_dict.get('invernet_epochs', 50),
    )
    return build_passive(s, seed=seed), s


def reconstruction_attack_v9(ckpt_path, epochs=50, target_resolution=64,
                             save_grid_path=None, grid_indices=None):
    try:
        import lpips as lpips_module
        from skimage.metrics import structural_similarity as ssim_fn
    except Exception as e:
        return {'error': f'lpips/skimage import failed: {e}', 'skipped': True}
    try:
        ckpt = _torch_load(ckpt_path)
        spec_dict = ckpt['spec']
        seed = ckpt['seed']
        passive, _ = _build_passive_for_load(spec_dict, seed=seed)
        passive = passive.to(DEVICE)
        passive.load_state_dict(ckpt['passive_state'], strict=True)
        passive.eval()
        use_quantizer = (spec_dict['passive_type'] == 'vq')

        target_tf = transforms.Compose([
            transforms.CenterCrop(CFG.input_size),
            transforms.Resize((target_resolution, target_resolution)),
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3),
        ])

        class PairDataset(Dataset):
            def __init__(self, images, feat_tf, target_tf):
                self.images = images; self.feat_tf = feat_tf; self.target_tf = target_tf
            def __len__(self): return len(self.images)
            def __getitem__(self, idx):
                pil = Image.fromarray(self.images[idx])
                return self.feat_tf(pil), self.target_tf(pil)

        tr_ds  = PairDataset(all_images[train_ind], val_transforms, target_tf)
        val_ds = PairDataset(all_images[val_ind],   val_transforms, target_tf)
        tr_loader  = DataLoader(tr_ds,  batch_size=64, shuffle=True,
                                num_workers=CFG.num_workers, pin_memory=True, drop_last=True)
        val_loader = DataLoader(val_ds, batch_size=64, shuffle=False,
                                num_workers=CFG.num_workers, pin_memory=True)

        with torch.no_grad():
            sample_img, _ = next(iter(tr_loader))
            emb, _, _, _  = passive(sample_img[:1].to(DEVICE), use_quantizer=use_quantizer)
            feat_dim = emb.shape[1]

        decoder = InverNetV9(feat_dim, target_resolution).to(DEVICE)
        opt   = optim.Adam(decoder.parameters(), lr=1e-3, weight_decay=1e-5)
        sched = CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-5)
        mse_crit = nn.MSELoss()
        lpips_loss_net = lpips_module.LPIPS(net='alex', verbose=False).to(DEVICE)
        for p in lpips_loss_net.parameters(): p.requires_grad = False
        lpips_loss_net.eval()

        for ep in range(1, epochs + 1):
            decoder.train()
            ep_l = 0.0; n = 0
            for img_feat, img_tgt in tr_loader:
                img_feat, img_tgt = img_feat.to(DEVICE), img_tgt.to(DEVICE)
                with torch.no_grad():
                    emb, _, _, _ = passive(img_feat, use_quantizer=use_quantizer)
                recon = decoder(emb)
                loss  = mse_crit(recon, img_tgt) + 0.1 * lpips_loss_net(recon, img_tgt).mean()
                opt.zero_grad(); loss.backward(); opt.step()
                ep_l += float(loss.item()); n += 1
            sched.step()
            if ep == 1 or ep % 10 == 0 or ep == epochs:
                print(f'  [InverNet] ep {ep:02d}/{epochs}  loss={ep_l/max(1,n):.4f}')

        decoder.eval()
        ssim_scores, lpips_scores = [], []
        from skimage.metrics import structural_similarity as ssim_fn
        with torch.no_grad():
            for img_feat, img_tgt in val_loader:
                img_feat, img_tgt = img_feat.to(DEVICE), img_tgt.to(DEVICE)
                emb, _, _, _ = passive(img_feat, use_quantizer=use_quantizer)
                recon = decoder(emb)
                lp = lpips_loss_net(recon, img_tgt).squeeze().detach().cpu().numpy()
                if lp.ndim == 0: lp = np.array([float(lp)])
                lpips_scores.extend(lp.tolist())
                r_np = ((recon.clamp(-1,1)+1)/2).permute(0,2,3,1).cpu().numpy()
                t_np = ((img_tgt.clamp(-1,1)+1)/2).permute(0,2,3,1).cpu().numpy()
                for b in range(r_np.shape[0]):
                    try:
                        s = ssim_fn(t_np[b], r_np[b], channel_axis=2, data_range=1.0,
                                    gaussian_weights=True, sigma=1.5, use_sample_covariance=False)
                        ssim_scores.append(float(s))
                    except Exception:
                        pass

        if save_grid_path and grid_indices:
            _save_grid_png(passive, decoder, grid_indices, save_grid_path,
                           target_tf=target_tf, use_quantizer=use_quantizer)

        return {
            'ssim_mean':  float(np.mean(ssim_scores))  if ssim_scores  else float('nan'),
            'ssim_std':   float(np.std(ssim_scores))   if ssim_scores  else float('nan'),
            'lpips_mean': float(np.mean(lpips_scores)) if lpips_scores else float('nan'),
            'lpips_std':  float(np.std(lpips_scores))  if lpips_scores else float('nan'),
            'feat_dim': int(feat_dim), 'invernet_epochs': int(epochs), 'skipped': False,
        }
    except Exception as e:
        return {'error': str(e), 'trace': traceback.format_exc(), 'skipped': True}


@torch.no_grad()
def _save_grid_png(passive, decoder, sample_indices, out_path, target_tf, use_quantizer):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    passive.eval(); decoder.eval()
    n   = len(sample_indices)
    fig, axes = plt.subplots(n, 3, figsize=(6, 2 * n))
    if n == 1: axes = axes.reshape(1, -1)
    col_titles = ['orig', 'recon', 'diff']
    for row, idx in enumerate(sample_indices):
        pil      = Image.fromarray(all_images[idx])
        feat_in  = val_transforms(pil).unsqueeze(0).to(DEVICE)
        tgt_in   = target_tf(pil).unsqueeze(0).to(DEVICE)
        emb, _, _, _ = passive(feat_in, use_quantizer=use_quantizer)
        recon    = decoder(emb)
        r_np = ((recon.clamp(-1,1)+1)/2).squeeze(0).permute(1,2,0).cpu().numpy()
        t_np = ((tgt_in.clamp(-1,1)+1)/2).squeeze(0).permute(1,2,0).cpu().numpy()
        diff = np.abs(r_np - t_np)
        for c, (img, title) in enumerate([
            (t_np, 'orig'), (r_np, 'recon'), (diff / max(diff.max(), 1e-6), 'diff')
        ]):
            axes[row, c].imshow(np.clip(img, 0, 1))
            axes[row, c].set_axis_off()
            if row == 0: axes[row, c].set_title(title, fontsize=10)
        axes[row, 0].set_ylabel(CLASS_NAMES[label_ints[idx]], fontsize=8, rotation=0,
                                labelpad=40, va='center')
    fig.tight_layout()
    fig.savefig(out_path, dpi=120, bbox_inches='tight')
    plt.close(fig)
    print(f'  Grid saved: {out_path}')

print('Reconstruction attack and grid saver defined.')

In [ ]:
def _pick_grid_indices():
    val_labels = label_ints[val_ind]
    chosen = []
    rng = np.random.RandomState(20260429)
    for c in tail_classes:
        cands = np.where(val_labels == c)[0]
        if len(cands) > 0:
            chosen.append(int(val_ind[cands[rng.randint(0, len(cands))]]))
    head_classes = [c for c in range(NUM_CLASSES) if c not in tail_classes]
    for c in head_classes[:2]:
        cands = np.where(val_labels == c)[0]
        if len(cands) > 0:
            chosen.append(int(val_ind[cands[rng.randint(0, len(cands))]]))
    return chosen[:6]

GRID_INDICES = _pick_grid_indices()
print(f'Grid indices ({len(GRID_INDICES)}): {GRID_INDICES}')
print(f'  classes: {[CLASS_NAMES[label_ints[i]] for i in GRID_INDICES]}')

In [ ]:
# ── Methods to generate grids for ──
# Ordered to tell the paper story row by row:
#   plain VFL (baseline) → sign quant (128b) → VQ at 128b → VQ K64 (best)
GRID_METHODS = [
    MethodSpec(name='A_plain_vfl',  passive_type='plain', comm_bits=40960),
    MethodSpec(name='S_sign_quant', passive_type='sign',  comm_bits=128),
    MethodSpec(name='H_vq_M16',     passive_type='vq',    comm_bits=128,
               vq_codebook_size=256, vq_num_subspaces=16),
    MethodSpec(name='H_vq_K64',     passive_type='vq',    comm_bits=48,
               vq_codebook_size=64,  vq_num_subspaces=8),
]

for spec in GRID_METHODS:
    for seed in CFG.seeds:
        ckpt_path = f'{CKPT_DIR}/{spec.name}_seed{seed}.pt'
        grid_path = f'{FIG_DIR}/recon_grid_{spec.name}_seed{seed}.png'
        if not os.path.exists(ckpt_path):
            print(f'[miss] {spec.name} seed={seed}: checkpoint not found'); continue
        print(f'\n>>> {spec.name}  seed={seed}')
        res = reconstruction_attack_v9(
            ckpt_path,
            epochs=spec.invernet_epochs,
            save_grid_path=grid_path,
            grid_indices=GRID_INDICES,
        )
        if res.get('skipped'):
            print(f'  [failed] {res.get("error")}')
        else:
            print(f'  SSIM={res["ssim_mean"]:.4f}  LPIPS={res["lpips_mean"]:.4f}')

print('\nDone. Download from: /kaggle/working/figs/recon_grid_*.png')
